# PYNQ-Z2 ANC Project - Audio Codec Interface Test

This notebook will:
1. Test the I2S codec interface on Pmod JA
2. Verify audio capture from the codec
3. Verify audio playback to the codec
4. Check the sample rate and clock generation

## Prerequisites
- Complete `01_setup.ipynb` successfully
- Connect an I2S audio codec to Pmod JA:
  - JA[0] (Y18): MCLK (12.5 MHz)
  - JA[1] (Y19): BCLK (~3.125 MHz)
  - JA[2] (Y16): LRCLK (~48.8 kHz)
  - JA[3] (Y17): SDATA_IN (ADC output)
  - JA[4] (U18): SDATA_OUT (DAC input)

## 1. Import Libraries and Load Overlay

In [ ]:
from pynq import Overlay
from pynq import MMIO
import numpy as np
import time

# Load the bitstream
overlay = Overlay('anc_project.bit')
print("Bitstream loaded successfully")

## 2. Access Codec Controller Registers

The audio codec controller should expose registers for:
- Left channel data (primary audio)
- Right channel data (reference noise)
- Output data (filtered audio)
- Sample valid flag

In [ ]:
# Find codec controller IP
codec_controller = None

for ip_name, ip_info in overlay.ip_dict.items():
    if 'codec' in ip_name.lower() or 'audio' in ip_name.lower():
        codec_controller = ip_name
        print(f"Found codec controller: {ip_name}")
        print(f"  Address: 0x{ip_info['phys_addr']:08X}")
        print(f"  Size: {ip_info['addr_range']} bytes")
        break

if codec_controller:
    mmio = MMIO(overlay.ip_dict[codec_controller]['phys_addr'], 
               overlay.ip_dict[codec_controller]['addr_range'])
    print(f"\nMMIO object created for {codec_controller}")
else:
    print("No codec controller found. Check if the design exposes codec registers.")

## 3. Generate Test Audio Signal

Create a simple sine wave for testing the audio path.

In [ ]:
import numpy as np

# Audio parameters
SAMPLE_RATE = 48000  # ~48.8 kHz actual
DURATION = 2.0  # seconds
FREQUENCY = 440  # A4 note

# Generate sine wave
t = np.linspace(0, DURATION, int(SAMPLE_RATE * DURATION), endpoint=False)
sine_wave = np.sin(2 * np.pi * FREQUENCY * t)

# Convert to 16-bit signed integer
sine_wave_16bit = (sine_wave * 32767).astype(np.int16)

print(f"Generated sine wave:")
print(f"  Duration: {DURATION} seconds")
print(f"  Sample rate: {SAMPLE_RATE} Hz")
print(f"  Frequency: {FREQUENCY} Hz")
print(f"  Samples: {len(sine_wave_16bit)}")
print(f"  Range: [{sine_wave_16bit.min()}, {sine_wave_16bit.max()}]")

## 4. Visualize Test Signal

In [ ]:
import matplotlib.pyplot as plt

# Plot first 1000 samples
plt.figure(figsize=(12, 4))
plt.plot(sine_wave_16bit[:1000])
plt.title('Test Sine Wave (First 1000 Samples)')
plt.xlabel('Sample')
plt.ylabel('Amplitude (16-bit)')
plt.grid(True)
plt.show()

## 5. Monitor Codec Clock Signals

Check if the codec is generating the expected clocks.
Note: This requires logic analyzer or oscilloscope for external verification.
We can infer clock activity from register updates.

In [ ]:
if codec_controller:
    # Monitor sample valid flag to infer clock activity
    SAMPLE_VALID_OFFSET = 0x10  # Adjust based on your design
    
    print("Monitoring sample valid flag for 2 seconds...")
    print("(Should toggle at ~48.8 kHz if clocks are running)")
    
    sample_count = 0
    start_time = time.time()
    
    while time.time() - start_time < 2.0:
        try:
            sample_valid = mmio.read(SAMPLE_VALID_OFFSET)
            if sample_valid & 1:
                sample_count += 1
        except:
            pass
        time.sleep(0.01)
    
    print(f"\nSamples detected in 2 seconds: {sample_count}")
    print(f"Expected (~48.8 kHz): {int(48800 * 2)}")
    
    if sample_count > 80000:
        print("✓ Clocks appear to be running correctly")
    else:
        print("✗ Clocks may not be running or register offset incorrect")
else:
    print("Skipping clock test - no codec controller found.")

## 6. Test Audio Capture (Left Channel)

Capture audio samples from the left channel (primary audio input).

In [ ]:
if codec_controller:
    LEFT_DATA_OFFSET = 0x00  # Adjust based on your design
    
    print("Capturing 1000 samples from left channel...")
    print("(Connect audio source to codec left input)")
    
    captured_samples = []
    start_time = time.time()
    
    while len(captured_samples) < 1000 and time.time() - start_time < 5.0:
        try:
            sample = mmio.read(LEFT_DATA_OFFSET)
            # Convert to signed 16-bit
            if sample >= 32768:
                sample = sample - 65536
            captured_samples.append(sample)
        except:
            pass
        time.sleep(0.001)
    
    captured_samples = np.array(captured_samples)
    print(f"\nCaptured {len(captured_samples)} samples")
    print(f"Range: [{captured_samples.min()}, {captured_samples.max()}]")
    print(f"Mean: {captured_samples.mean():.2f}")
    print(f"Std: {captured_samples.std():.2f}")
else:
    print("Skipping capture test - no codec controller found.")

## 7. Visualize Captured Audio

In [ ]:
if codec_controller and len(captured_samples) > 0:
    plt.figure(figsize=(12, 4))
    plt.plot(captured_samples)
    plt.title('Captured Audio (Left Channel)')
    plt.xlabel('Sample')
    plt.ylabel('Amplitude (16-bit)')
    plt.grid(True)
    plt.show()
else:
    print("No samples to display.")

## 8. Test Audio Playback

Send test audio to the codec for playback.

In [ ]:
if codec_controller:
    OUTPUT_DATA_OFFSET = 0x08  # Adjust based on your design
    
    print("Playing 1 second of sine wave...")
    print("(Connect headphones/speakers to codec output)")
    
    samples_to_play = int(SAMPLE_RATE * 1.0)
    start_time = time.time()
    
    for i in range(samples_to_play):
        sample = int(sine_wave_16bit[i])
        # Convert to unsigned 16-bit for register write
        if sample < 0:
            sample = sample + 65536
        try:
            mmio.write(OUTPUT_DATA_OFFSET, sample)
        except:
            pass
        # Wait for approximately sample period
        time.sleep(1.0 / SAMPLE_RATE)
    
    print("\nPlayback complete.")
else:
    print("Skipping playback test - no codec controller found.")

## 9. Test Stereo Capture (Both Channels)

Capture both left and right channels simultaneously.

In [ ]:
if codec_controller:
    LEFT_DATA_OFFSET = 0x00
    RIGHT_DATA_OFFSET = 0x04  # Adjust based on your design
    
    print("Capturing 500 samples from both channels...")
    
    left_samples = []
    right_samples = []
    start_time = time.time()
    
    while len(left_samples) < 500 and time.time() - start_time < 5.0:
        try:
            left_sample = mmio.read(LEFT_DATA_OFFSET)
            right_sample = mmio.read(RIGHT_DATA_OFFSET)
            
            # Convert to signed
            if left_sample >= 32768:
                left_sample = left_sample - 65536
            if right_sample >= 32768:
                right_sample = right_sample - 65536
            
            left_samples.append(left_sample)
            right_samples.append(right_sample)
        except:
            pass
        time.sleep(0.001)
    
    left_samples = np.array(left_samples)
    right_samples = np.array(right_samples)
    
    print(f"\nCaptured {len(left_samples)} samples from each channel")
    print(f"Left channel range: [{left_samples.min()}, {left_samples.max()}]")
    print(f"Right channel range: [{right_samples.min()}, {right_samples.max()}]")
else:
    print("Skipping stereo capture - no codec controller found.")

## 10. Visualize Stereo Capture

In [ ]:
if codec_controller and len(left_samples) > 0:
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6))
    
    ax1.plot(left_samples)
    ax1.set_title('Left Channel (Primary Audio)')
    ax1.set_ylabel('Amplitude')
    ax1.grid(True)
    
    ax2.plot(right_samples)
    ax2.set_title('Right Channel (Reference Noise)')
    ax2.set_xlabel('Sample')
    ax2.set_ylabel('Amplitude')
    ax2.grid(True)
    
    plt.tight_layout()
    plt.show()
else:
    print("No stereo samples to display.")

## 11. Check Clock Frequency

Verify the codec is operating at the expected sample rate by measuring sample arrival rate.

In [ ]:
if codec_controller:
    print("Measuring sample rate...")
    
    samples = 0
    start_time = time.time()
    
    while samples < 10000:
        try:
            _ = mmio.read(LEFT_DATA_OFFSET)
            samples += 1
        except:
            pass
    
    elapsed = time.time() - start_time
    measured_rate = samples / elapsed
    
    print(f"Samples captured: {samples}")
    print(f"Time elapsed: {elapsed:.3f} seconds")
    print(f"Measured sample rate: {measured_rate:.1f} Hz")
    print(f"Expected sample rate: ~48828 Hz")
    
    if 45000 < measured_rate < 55000:
        print("✓ Sample rate within expected range")
    else:
        print("✗ Sample rate outside expected range")
else:
    print("Skipping sample rate test - no codec controller found.")

## 12. Summary

### Verification Checklist:
- [ ] Codec controller accessible
- [ ] Clock signals active (inferred from sample valid)
- [ ] Left channel capture working
- [ ] Right channel capture working
- [ ] Audio playback working
- [ ] Sample rate ~48.8 kHz

### Next Steps:
- Proceed to `03_anc_operation.ipynb` to test the full ANC system
- Ensure audio sources are connected:
  - Primary audio (signal + noise) → Left channel
  - Reference noise (correlated) → Right channel
- Verify noise cancellation with real audio